# EXP01 — Reprodução do artigo original

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


## 2. Instalar as dependências

Instalamos as bibliotecas usadas nas análises. Se estiver usando um ambiente local já configurado, a instalação pode ser ignorada.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Baixar e conferir os dados

Os arquivos processados do EXP01 e EXP02 estão disponíveis para **leitura pública** no Google Drive da pesquisa (links em `docs/DADOS.md`). A célula seguinte tenta baixar os dois Parquets e compara seus hashes SHA-256 com os registros históricos. Se o Google Drive bloquear o download, no Colab você poderá enviar os arquivos manualmente.

Dados já presentes com hashes corretos são reutilizados. A verificação de integridade não valida a qualidade dos rótulos nem refaz o processamento bruto da blockchain.


In [ ]:
from pathlib import Path
import hashlib
import tempfile
import os
import gdown

# Arquivos processados, publicados para leitura pela equipe da pesquisa.
ARQUIVOS = {
    "features_treino.parquet": {
        "id": "1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S",
        "sha256": "0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32",
    },
    "features_teste_20201029.parquet": {
        "id": "1g22eCccxC4v_S2RmxPhFZyzfF1ARuzPk",
        "sha256": "6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51",
    },
}

pasta_dados = RAIZ / "data" / "processed"
pasta_dados.mkdir(parents=True, exist_ok=True)

def conferir_hash(caminho, esperado):
    resumo = hashlib.sha256()
    with caminho.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            resumo.update(bloco)
    if resumo.hexdigest() != esperado:
        raise ValueError(f"O arquivo {caminho.name} não corresponde ao SHA-256 esperado.")

def baixar_e_conferir(nome, info):
    destino = pasta_dados / nome
    if destino.is_file():
        conferir_hash(destino, info["sha256"])
        print(f"Arquivo local conferido: {nome}")
        return

    temporario = pasta_dados / (nome + ".download")
    try:
        print(f"Baixando {nome} do Google Drive público...")
        resultado = gdown.download(id=info["id"], output=str(temporario), quiet=False)
        if not resultado or not temporario.is_file():
            raise RuntimeError("O Drive não forneceu o arquivo.")
        conferir_hash(temporario, info["sha256"])
        temporario.replace(destino)
        print(f"Download conferido: {nome} ({destino.stat().st_size} bytes)")
    finally:
        temporario.unlink(missing_ok=True)

falhas = []
for nome, info in ARQUIVOS.items():
    try:
        baixar_e_conferir(nome, info)
    except Exception as erro:
        if (pasta_dados / nome).exists():
            raise  # Não ignorar arquivo local com hash incorreto.
        print(f"Não foi possível baixar {nome}: {erro}")
        falhas.append(nome)

if falhas:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Download indisponível. Baixe os arquivos indicados em docs/DADOS.md "
            "e coloque-os em data/processed/."
        ) from exc
    print("Selecione os arquivos que faltaram:", ", ".join(falhas))
    recebidos = files.upload()
    for nome in falhas:
        if nome not in recebidos:
            raise FileNotFoundError(f"Upload não incluiu {nome}")
        destino = pasta_dados / nome
        destino.write_bytes(recebidos[nome])
        try:
            conferir_hash(destino, ARQUIVOS[nome]["sha256"])
        except Exception:
            destino.unlink(missing_ok=True)
            raise
        print(f"Upload conferido: {nome}")

print("Todos os dados necessários foram conferidos.")


## 3. Carregar os dados

Precisamos de `features_treino.parquet` e `features_teste_20201029.parquet` em `data/processed/`. Esses arquivos **não** são baixados do Drive privado. Confira `docs/DADOS.md` e não troque os dados por uma amostra sintética para declarar reprodução.

In [ ]:
from pathlib import Path
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados, verificar_referencia

PASTA = RAIZ / "data" / "processed"
ARQ_TREINO = PASTA / "features_treino.parquet"
ARQ_2020 = PASTA / "features_teste_20201029.parquet"

for arquivo in [ARQ_TREINO, ARQ_2020]:
    print(verificar_referencia(arquivo))

treino = abrir_dados(ARQ_TREINO, ["label", "n_tx", *FEATURES])
teste = abrir_dados(ARQ_2020, ["recipient", "n_tx", "tornado", *FEATURES])
print("Observações de treino:", len(treino), "| Destinatários no teste:", len(teste))

## 4. Reproduzir o experimento

Random Forest com 100 árvores, profundidade máxima 5 e semente 42. O balanceamento é feito no treino. A seleção do teste usa pelo menos dez transações e o alerta corresponde a `score > 0,50`. A validação em cinco divisões é **interna**; o contrato positivo se repete entre treino e validação.

In [ ]:
from ethmix.execucao import treinar, prever

modelo, validacao = treinar(treino, seed=42, min_tx=10)
predicoes = prever(modelo, teste, min_tx=10, limiar=0.50)
alertas = predicoes.loc[predicoes["alerta"]].copy()

print("Média das métricas internas:")
print(validacao[["precision", "recall", "f1"]].mean().round(4))
print("Destinatários elegíveis:", len(predicoes))
print("Alertas:", len(alertas))
print("Contratos conhecidos sinalizados:", alertas["tornado"].notna().sum())
display(alertas[["recipient", "score", "n_tx"]].sort_values("score", ascending=False).head(20))

## 5. Salvar e conferir

A referência histórica é **9 alertas e 3/3 pools conhecidos** em 29/10/2020. Se a execução gerar outro valor, registre a diferença: não mude os parâmetros apenas para alcançar o número publicado.

In [ ]:
from datetime import datetime, timezone
from ethmix.execucao import guardar_arquivos

quando = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
saida = RAIZ / "results" / "generated" / "EXP01" / quando
manifesto = {"experimento": "EXP01", "min_tx": 10, "limiar": ">0.50",
             "seed": 42, "observacoes_treino": len(treino),
             "destinatarios_elegiveis": len(predicoes), "alertas": len(alertas)}
guardar_arquivos(saida, modelo, validacao, alertas, manifesto)
print("Arquivos salvos em:", saida)

## 6. O que podemos concluir?

O número de alertas e a recuperação dos pools conhecidos são resultados observados no recorte. Não permitem calcular precisão populacional nem provar que o detector identifica outros protocolos inéditos.